# FLAG 2027 - Notebook 18: VoxCeleb2 + new voice encoders + recipe v2 (step B of ARCHITECTURE_v2)

Notebook 17: ReDimNet2 + audeering gave -1.17 CV (26.70 -> 25.53) without VoxCeleb. Here:

1. Re-download the two VoxCeleb2 zips, rebuild EXACTLY the same clips as Notebooks 07/13 (alignment check on recomputed ECAPA), extract **ReDimNet2** and **audeering** for them.
2. Train with ALL data (challenge + MAV v1/v2/v3 + VoxCeleb2) and the R4 feature set, four recipe variants:
   - V0: P1 recipe (bal75 batches 16x8, fixed prototypes)  <- reference, same as dev 29.02 but with new features
   - V1: batch 64 ids x 8, cosine LR
   - V2: V1 + domain-balanced hard-identity mining (after 10 warm-up epochs)
   - V3: V2 + scale 16 and CosFace margin 0.25 on the fixed-prototype logits
3. ZIPs: V0 and the best variant; plus a raw-average ensemble of the best variant with P1 (Notebook 15 dev scores).

First run about 4-5 h (download ~40 min, ReDimNet2 ~2.5 h, training ~1 h). Resumable per zip and per training run. Paste the **SUMMARY** to Claude.

In [ ]:
# Cell 1 - Drive, GPU, packages, HF token
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install speechbrain huggingface_hub
import torch, shutil
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
HF_TOKEN = userdata.get('HF_TOKEN')
print('Device:', dev, '| HF token found:', HF_TOKEN is not None, '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))

In [ ]:
# Cell 2 - paths + encoders (ECAPA check, language-ID, ReDimNet2, audeering) - identical definitions to Notebooks 13/17
import os, re, json, time, zipfile, copy, subprocess
import numpy as np, pandas as pd
import librosa, soundfile as sf
import torch.nn as nn
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027')
FEATDIR = os.path.join(PROJECT, 'features')
RESDIR = os.path.join(PROJECT, 'results')
CACHE = os.path.join(PROJECT, 'models_cache')
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
try:
    from speechbrain.inference.interfaces import Pretrained
    from speechbrain.inference.classifiers import EncoderClassifier
except ImportError:
    from speechbrain.pretrained.interfaces import Pretrained
    from speechbrain.pretrained import EncoderClassifier
class Encoder(Pretrained):
    MODULES_NEEDED = ['compute_features', 'mean_var_norm', 'embedding_model']
    def encode_batch(self, wavs, wav_lens=None):
        if wav_lens is None:
            wav_lens = torch.ones(wavs.shape[0], device=self.device)
        wavs, wav_lens = wavs.to(self.device).float(), wav_lens.to(self.device)
        return self.mods.embedding_model(self.mods.mean_var_norm(self.mods.compute_features(wavs), wav_lens), wav_lens)
ECAPA = Encoder.from_hparams(source='yangwang825/ecapa-tdnn-vox2', savedir=os.path.join(CACHE, 'ecapa_vox2'), run_opts={'device': dev}); ECAPA.eval()
LID = EncoderClassifier.from_hparams(source='speechbrain/lang-id-voxlingua107-ecapa', savedir=os.path.join(CACHE, 'lid_voxlingua107'), run_opts={'device': dev})
def batches_by_length(paths, bs):
    order = np.argsort([sf.info(p).duration for p in paths])
    for i in range(0, len(order), bs):
        yield order[i:i + bs]
def pad(paths, idx, max_sec=None):
    wavs = [librosa.load(paths[j], sr=16000, mono=True)[0] for j in idx]
    if max_sec:
        wavs = [w[:16000 * max_sec] for w in wavs]
    lens = torch.tensor([len(w) for w in wavs], dtype=torch.float); L = int(lens.max())
    b = torch.zeros(len(wavs), L)
    for k, w in enumerate(wavs):
        b[k, :len(w)] = torch.from_numpy(w)
    return b, lens / L
def enc_ecapa(paths, bs=16):
    out = np.zeros((len(paths), 192), np.float32)
    for idx in tqdm(list(batches_by_length(paths, bs)), desc='ecapa (check)'):
        b, rl = pad(paths, idx)
        with torch.no_grad():
            out[idx] = ECAPA.encode_batch(b, rl).squeeze(1).cpu().numpy()
    return out
def language_of(paths, bs=32):
    out = [''] * len(paths)
    for idx in tqdm(list(batches_by_length(paths, bs)), desc='language-ID'):
        b, rl = pad(paths, idx, max_sec=10)
        with torch.no_grad():
            labs = LID.classify_batch(b.to(dev), rl.to(dev))[3]
        for j, lab in zip(idx, labs):
            out[j] = lab.split(':')[0].strip()
    return np.array(out)
def load_redimnet():
    tries = [('PalabraAI/redimnet2', dict(model_name='b6', train_type='lm', dataset='vb2+vox2_v0', pretrained=True), 'redimnet2'),
             ('PalabraAI/redimnet2', dict(model_name='b6', train_type='lm', dataset='vox2', pretrained=True), 'redimnet2'),
             ('IDRnD/ReDimNet', dict(model_name='b6', train_type='ft_lm', dataset='vox2'), 'ReDimNet')]
    for repo, kw, entry in tries:
        try:
            m = torch.hub.load(repo, entry, **kw).to(dev).eval(); note('ReDimNet loaded: %s %s' % (repo, kw)); return m
        except Exception as e:
            print('  failed', repo, kw, '->', repr(e)[:160])
    raise RuntimeError('no ReDimNet variant could be loaded')
def enc_redim(paths, bs=16, max_sec=10):
    net = load_redimnet(); order = np.argsort([sf.info(p).duration for p in paths]); emb = {}
    for i in tqdm(range(0, len(order), bs), desc='redimnet'):
        idx = order[i:i + bs]
        wavs = [librosa.load(paths[j], sr=16000, mono=True)[0][:16000 * max_sec] for j in idx]
        L = max(len(w) for w in wavs); b = torch.zeros(len(wavs), L)
        for k_, w in enumerate(wavs):
            b[k_] = torch.from_numpy(np.tile(w, int(np.ceil(L / max(len(w), 1))))[:L])
        with torch.no_grad():
            e = net(b.to(dev))
        e = e[0] if isinstance(e, (list, tuple)) else e
        for k_, j in enumerate(idx):
            emb[int(j)] = e[k_].float().cpu().numpy().reshape(-1)
    return np.stack([emb[j] for j in range(len(paths))])
from transformers import Wav2Vec2Processor, Wav2Vec2Model, Wav2Vec2PreTrainedModel
class ModelHead(nn.Module):
    def __init__(self, hidden, num_labels, drop):
        super().__init__()
        self.dense = nn.Linear(hidden, hidden); self.dropout = nn.Dropout(drop); self.out_proj = nn.Linear(hidden, num_labels)
    def forward(self, x):
        return self.out_proj(self.dropout(torch.tanh(self.dense(self.dropout(x)))))
class AgeGenderModel(Wav2Vec2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.wav2vec2 = Wav2Vec2Model(config)
        self.age = ModelHead(config.hidden_size, 1, config.final_dropout); self.gender = ModelHead(config.hidden_size, 3, config.final_dropout)
        self.post_init()
    def forward(self, input_values):
        h = self.wav2vec2(input_values)[0].mean(1)
        return h, self.age(h), torch.softmax(self.gender(h), dim=1)
def load_audeer():
    name = 'audeering/wav2vec2-large-robust-24-ft-age-gender'
    proc = Wav2Vec2Processor.from_pretrained(name)
    try:
        model = AgeGenderModel.from_pretrained(name)
    except Exception as e:
        print('  from_pretrained failed (%r) -> manual load' % (repr(e)[:120],))
        base = Wav2Vec2Model.from_pretrained(name); model = AgeGenderModel(base.config); model.wav2vec2 = base
        try:
            from safetensors.torch import load_file; sd = load_file(hf_hub_download(name, 'model.safetensors'))
        except Exception:
            sd = torch.load(hf_hub_download(name, 'pytorch_model.bin'), map_location='cpu')
        model.load_state_dict({k: v for k, v in sd.items() if k.startswith(('age.', 'gender.'))}, strict=False)
    return proc, model.to(dev).eval().half()
def enc_audeer(paths, bs=16, max_sec=8):
    proc, model = load_audeer(); H, S = [], []
    for i in tqdm(range(0, len(paths), bs), desc='audeering'):
        wavs = [librosa.load(p, sr=16000, mono=True)[0][:16000 * max_sec] for p in paths[i:i + bs]]
        x = proc(wavs, sampling_rate=16000, padding=True, return_tensors='pt')['input_values'].to(dev).half()
        with torch.no_grad():
            h, a, g = model(x)
        H.append(h.float().cpu().numpy()); S.append(torch.cat([a, g], 1).float().cpu().numpy())
    return np.concatenate(H), np.concatenate(S)
print('encoders ready')

In [ ]:
# Cell 3 - rebuild the SAME VoxCeleb2 clips (Notebook 07 selection) and extract ReDimNet2 + audeering (RESUMABLE per zip)
ZIPS = [('vox2/vox2_test_mp4.zip', 40), ('vox2/vox2_mp4_1.zip', 20)]
def ffmpeg_audio(job):
    mp4, wav, jpg = job
    a = subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-i', mp4, '-vn', '-ac', '1', '-ar', '16000', wav]).returncode
    v = subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-ss', '1.0', '-i', mp4, '-frames:v', '1', '-q:v', '2', jpg]).returncode
    return a == 0 and v == 0 and os.path.exists(wav) and os.path.exists(jpg)
for zname, cpi in ZIPS:
    tag = os.path.basename(zname).replace('.zip', '')
    out_r, out_a = os.path.join(FEATDIR, 'vox_' + tag + '_redim.npy'), os.path.join(FEATDIR, 'vox_' + tag + '_audeer.npz')
    if os.path.exists(out_r) and os.path.exists(out_a):
        print(tag, 'already done'); continue
    OLD = dict(np.load(os.path.join(FEATDIR, 'vox_' + tag + '.npz')))
    t0 = time.time()
    zpath = hf_hub_download('ProgramComputer/voxceleb', zname, repo_type='dataset', token=HF_TOKEN, local_dir='/content/vox')
    zf = zipfile.ZipFile(zpath); tree = {}
    for n in zf.namelist():
        mm = re.search(r'(id\d{5})/([^/]+)/([^/]+\.mp4)$', n)
        if mm:
            tree.setdefault(mm.group(1), {}).setdefault(mm.group(2), []).append(n)
    rng = np.random.default_rng(0); picked = []
    for pid, vids in tree.items():
        pools = [list(rng.permutation(v)) for v in vids.values()]; chosen = []
        while len(chosen) < cpi and any(pools):
            for p in pools:
                if p and len(chosen) < cpi:
                    chosen.append(p.pop())
        picked += [(pid, c) for c in chosen]
    W = '/content/vox_work/' + tag; os.makedirs(W, exist_ok=True); jobs = []
    for i, (pid, member) in enumerate(picked):
        jobs.append((zf.extract(member, W + '/mp4'), '%s/%06d.wav' % (W, i), '%s/%06d.jpg' % (W, i)))
    zf.close(); os.remove(zpath)
    with ThreadPoolExecutor(8) as ex:
        ok = list(tqdm(ex.map(ffmpeg_audio, jobs), total=len(jobs), desc='ffmpeg'))
    keep = [i for i, g in enumerate(ok) if g]
    wavs = [jobs[i][1] for i in keep]; spk = np.array(['vox_' + picked[i][0] for i in keep])
    lang = language_of(wavs); nb = lang != 'bn'
    wavs = [w for w, k in zip(wavs, nb) if k]; spk = spk[nb]
    assert len(spk) == len(OLD['spk']) and (spk == OLD['spk']).all(), 'row order differs from Notebook 07 - tell Claude'
    probe = np.random.default_rng(1).choice(len(wavs), 300, replace=False)
    e_new = enc_ecapa([wavs[j] for j in probe]); e_old = OLD['ecapa'][probe]
    cos = (e_new * e_old).sum(1) / (np.linalg.norm(e_new, axis=1) * np.linalg.norm(e_old, axis=1) + 1e-9)
    note('%s: %d rows | alignment median cosine %.4f' % (tag, len(spk), np.median(cos)))
    assert np.median(cos) > 0.98, 'rows do not line up - stop and tell Claude'
    if not os.path.exists(out_r):
        np.save(out_r, enc_redim(wavs).astype(np.float32)); note('%s: ReDimNet2 done (%.0f min)' % (tag, (time.time() - t0) / 60))
    if not os.path.exists(out_a):
        H, S = enc_audeer(wavs); np.savez(out_a, h=H.astype(np.float32), ag=S.astype(np.float32)); note('%s: audeering done (%.0f min)' % (tag, (time.time() - t0) / 60))
    shutil.rmtree(W, ignore_errors=True)

In [ ]:
# Cell 4 - load ALL data with the R4 feature set: face CLIP | voice ECAPA + WavLM + ResNet293 + ReDimNet2 + audeering(1024)
import torch.nn.functional as Fn
from sklearn.metrics import roc_curve
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
LOCAL = '/content/flag_small'; os.makedirs(LOCAL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    flag = os.path.join(LOCAL, z + '.done')
    if not os.path.exists(flag):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(LOCAL, members=[m for m in zf.namelist() if m.endswith(('.csv', '.txt'))])
        open(flag, 'w').close()
INDEX = [os.path.join(r, f) for r, _, fs in os.walk(LOCAL) for f in fs if f.endswith(('.csv', '.txt'))]
def find_one(*parts):
    hits = [p for p in INDEX if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
VOICE = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']
FEAT = {}
Xv, y = read_feat(find_one('train_English_voices.csv'), 192); FEAT['ecapa'] = {'train': Xv}
for s in SPLIT_NAMES[1:]:
    track, lang_ = s.rsplit('_', 1); FEAT['ecapa'][s] = read_feat(find_one('/' + track + '/', lang_ + '_test_voices.csv'), 192)[0]
for name in ['clip', 'wavlm_sv', 'r293', 'redim', 'audeer']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files if not s.endswith('_ag')}
y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz'))); EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
MAV['r293'], MAV['redim'], MAV['audeer'] = FEAT['r293']['mav'], FEAT['redim']['mav'], FEAT['audeer']['mav']
EXTRA['redim'], EXTRA['audeer'] = FEAT['redim']['extra'], FEAT['audeer']['extra']
VOX = {k: [] for k in ['spk', 'clip', 'ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']}
for zname, _ in ZIPS:
    tag = os.path.basename(zname).replace('.zip', '')
    d = dict(np.load(os.path.join(FEATDIR, 'vox_' + tag + '.npz')))
    d['r293'] = np.load(os.path.join(FEATDIR, 'vox_' + tag + '_r293.npy')); d['redim'] = np.load(os.path.join(FEATDIR, 'vox_' + tag + '_redim.npy'))
    d['audeer'] = np.load(os.path.join(FEATDIR, 'vox_' + tag + '_audeer.npz'))['h']
    for k in VOX:
        VOX[k].append(d[k])
VOX = {k: np.concatenate(v) for k, v in VOX.items()}
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def face_of(c):
    return l2(c).astype(np.float32)
def voice_of(src):
    return np.concatenate([l2(src[n]) for n in VOICE], 1).astype(np.float32)
CH = {n: FEAT[n]['train'] for n in ['clip'] + VOICE}
N_CH, N_MAV, N_EX, N_VOX = len(y), len(MAV['spk']), len(EXTRA['spk']), len(VOX['spk'])
ext_spk = list(MAV['spk']) + list(EXTRA['spk']) + list(VOX['spk'])
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.concatenate([y, np.array([ext_lab[s] for s in ext_spk])])
D_ALL = np.r_[np.zeros(N_CH, int), np.ones(N_MAV + N_EX, int), 2 * np.ones(N_VOX, int)]
F_RAW = np.concatenate([face_of(CH['clip']), face_of(MAV['clip']), face_of(EXTRA['clip']), face_of(VOX['clip'])])
V_RAW = np.concatenate([voice_of(CH), voice_of(MAV), voice_of(EXTRA), voice_of(VOX)])
EXT = N_CH + np.arange(N_MAV + N_EX + N_VOX)
note('Table: %d rows, %d people (in-domain %d), face %d-d, voice %d-d' % (len(Y_ALL), len(set(Y_ALL)), len(set(Y_ALL[D_ALL < 2])), F_RAW.shape[1], V_RAW.shape[1]))

In [ ]:
# Cell 5 - model with fixed prototypes (+ optional scale/margin), validation folds, recipe-v2 training
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed)
    spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5
rng = np.random.default_rng(42); folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P; P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class Branch(nn.Module):
    def __init__(self, d_in, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128, margin=False):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class)); self.margin = margin
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv)
        return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv, lab=None):
        ft, vt, f, v = self.embed(xf, xv)
        a = torch.sigmoid(self.att(torch.cat([f, v], 1))); fused = ft * a + (1 - a) * vt
        if self.margin:   # CosFace on fixed prototypes: s * (cos - m on the true class)
            cosine = Fn.normalize(self.cls(fused), dim=1) @ self.P.t()
            if lab is not None:
                cosine = cosine - 0.25 * Fn.one_hot(lab, cosine.shape[1]).float()
            return fused, 16.0 * cosine
        return fused, self.cls(fused) @ self.P.t()
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t()
    same = (lab[:, None] == lab[None, :]).float(); pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm
    ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def domain_normalize(tr_idx):
    F, V = F_RAW.copy(), V_RAW.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_RAW[trd].mean(0), F_RAW[trd].std(0) + 1e-6, V_RAW[trd].mean(0), V_RAW[trd].std(0) + 1e-6)
        F[alld] = (F_RAW[alld] - st[0]) / st[1]; V[alld] = (V_RAW[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch

SEEDS = [1, 2, 3]
K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS, IN_FRAC, WARM = 8, 2.0, 1e-4, 100, 15, 60, 0.75, 10
@torch.no_grad()
def class_means(model, XF, XV, tr, ytr_np, n_cls):
    model.eval(); mf, sf_, mv, sv = model.norm
    ft, vt, _, _ = model.embed((XF[tr] - mf) / sf_, (XV[tr] - mv) / sv)
    z = Fn.normalize(ft + vt, dim=1)
    M = torch.zeros(n_cls, z.shape[1], device=dev).index_add_(0, torch.tensor(ytr_np, device=dev), z)
    return Fn.normalize(M, dim=1)
def train_variant(tr_idx, cfg, vsets=None, seed=1, max_epochs=MAX_EP):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    P_ID = cfg['p_id']
    all_tr = np.concatenate([tr_idx, EXT])
    F, V, ch = domain_normalize(all_tr)
    XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev)
    cls_in = np.zeros(len(cl), bool); cls_in[np.unique(ytr_np[D_ALL[all_tr] < 2])] = True
    in_cls, out_cls = np.where(cls_in)[0], np.where(~cls_in)[0]; n_in = int(round(P_ID * IN_FRAC))
    pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev)
    model = FOP(len(cl), XF.shape[1], XV.shape[1], d=cfg['dim'], margin=cfg['margin']).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6)
    mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs * STEPS) if cfg['cosine'] else None
    best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    means = None
    for ep in range(1, max_epochs + 1):
        model.train()
        if cfg['hard'] and ep > WARM:
            means = class_means(model, XF, XV, tr, ytr_np, len(cl)); S = means @ means.t(); S.fill_diagonal_(-2)
            model.train()
        for _ in range(STEPS):
            if means is None:
                chosen = np.r_[rng.choice(in_cls, n_in, replace=False), rng.choice(out_cls, P_ID - n_in, replace=False)]
            else:   # domain-balanced hard mining: half seeds, half nearest neighbours from the same pool
                chosen = []
                for pool, n in [(in_cls, n_in), (out_cls, P_ID - n_in)]:
                    seeds = rng.choice(pool, n // 2, replace=False); used = set(seeds.tolist())
                    for s_ in seeds:
                        nb = torch.topk(S[s_, torch.tensor(pool, device=dev)], 6).indices.cpu().numpy()
                        for c in pool[nb]:
                            if c not in used:
                                used.add(c); break
                    chosen += list(used)
                chosen = np.array(chosen)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv, ytr[j])
            loss = Fn.cross_entropy(logits, ytr[j]) + ALPHA * opl_loss(fused, ytr[j])
            opt.zero_grad(); loss.backward(); opt.step()
            if sched is not None:
                sched.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch

VARIANTS = {'V0_p1_recipe':        dict(p_id=16, dim=128, cosine=False, hard=False, margin=False),
            'V1_batch64_cosine':   dict(p_id=64, dim=128, cosine=True,  hard=False, margin=False),
            'V2_+hardmining':      dict(p_id=64, dim=128, cosine=True,  hard=True,  margin=False),
            'V3_+margin_dim512':   dict(p_id=64, dim=512, cosine=True,  hard=True,  margin=True)}
CSV = os.path.join(RESDIR, 'nb18_cv.csv')
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['variant', 'seed', 'fold'])
t0 = time.time()
for name, cfg in VARIANTS.items():
    for seed in SEEDS:
        for k in range(K):
            if ((done.variant == name) & (done.seed == seed) & (done.fold == k)).any():
                continue
            tr_idx, vsets = FOLD_DATA[k]
            model, best_ep, XF, XV, _ = train_variant(tr_idx, cfg, vsets, seed=seed)
            e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
            row = pd.DataFrame([dict(variant=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
            row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False); done = pd.concat([done, row])
            print('%-20s seed %d fold %d | ep %3d | std %.2f | gender %.2f | %.0f min' % (name, seed, k, best_ep, e_std, e_gen, (time.time() - t0) / 60))
R = pd.read_csv(CSV); R['overall'] = (R['std'] + R['gender']) / 2
tab = R.groupby(['variant', 'seed'])[['std', 'gender', 'overall']].mean().groupby('variant').mean()
tab = tab.loc[[v for v in VARIANTS if v in tab.index]].round(2)
note('RECIPE v2 with R4 features + VoxCeleb (CV 5-fold x 3 seeds; P1 old features = 26.13, R4 no-Vox = 25.53)')
note(tab.to_string())
BEST = tab['overall'].idxmin(); note('BEST: %s (%.2f)' % (BEST, tab.loc[BEST, 'overall']))

In [ ]:
# Cell 6 - final models -> ZIPs (V0 and BEST) + raw-average ensemble with P1 (Notebook 15 dev scores)
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
def write_zip(name, scores):
    OUT = '/content/sub_' + name
    for track in ['no_gender', 'gender']:
        os.makedirs(os.path.join(OUT, track), exist_ok=True)
        for lang_, fname in LANG_FILE.items():
            split = track + '_' + lang_
            pair_ids = [l.split()[0] for l in open(find_one('/' + track + '/', lang_ + '_test.txt')) if l.strip()]
            assert len(pair_ids) == len(scores[split]) and np.isfinite(scores[split]).all()
            with open(os.path.join(OUT, track, fname), 'w') as f:
                for pid, v in zip(pair_ids, -scores[split]):
                    f.write('%s %.6f\n' % (pid, v))
    zp = os.path.join(SUBDIR, name + '.zip')
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
        for track in ['no_gender', 'gender']:
            for fname in LANG_FILE.values():
                zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
    note('Submission ZIP saved: ' + zp[len(BASE):])
def dev_scores(name):
    cfg = VARIANTS[name]; ep = int(np.median(R[R.variant == name]['best_ep'])); finals = []
    for s in SEEDS:
        m, _, _, _, ch = train_variant(np.arange(N_CH), cfg, None, seed=s, max_epochs=ep); finals.append(m)
    sc = {}
    for track in ['no_gender', 'gender']:
        for lang_ in LANG_FILE:
            split = track + '_' + lang_
            fx = ((face_of(FEAT['clip'][split]) - ch[0]) / ch[1]).astype(np.float32)
            vx = ((voice_of({n: FEAT[n][split] for n in VOICE}) - ch[2]) / ch[3]).astype(np.float32)
            xf = torch.tensor(fx, device=dev); xv = torch.tensor(vx, device=dev)
            sc[split] = np.mean([same_score(m, xf, xv) for m in finals], axis=0)
    np.savez(os.path.join(RESDIR, 'devscores_18_%s.npz' % name), **sc); return sc
S0 = dev_scores('V0_p1_recipe'); write_zip('18_V0_R4feat_vox', S0)
if BEST != 'V0_p1_recipe':
    SB = dev_scores(BEST); write_zip('18_' + BEST, SB)
else:
    SB = S0
p15 = os.path.join(RESDIR, 'devscores_15_P1_proto.npz')
if os.path.exists(p15):
    P1 = dict(np.load(p15)); write_zip('18_ENS_best+P1', {s: (SB[s] + P1[s]) / 2 for s in SB})
print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)